In [266]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [267]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df[['Serangan_Jantung']]

In [268]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [269]:
params = {
    'C': np.logspace(-5, -2, 100),
    'penalty': ['l2'],
    'max_iter': [500, 1000, 2000],
    'solver': ['lbfgs']
}
log_reg = LogisticRegression()
random_search = RandomizedSearchCV(estimator=log_reg,param_distributions=params,
                                   n_iter=20,cv=5,scoring='accuracy',
                                   random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train.values.ravel())

print(f'Hyperparameter Terbaik: {random_search.best_params_}')
print(f'Akurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')

best_model = random_search.best_estimator_
tes_accuracy = best_model.score(X_test, y_test)

y_pred_test= best_model.predict(X_test)
y_prob_test= best_model.predict_proba(X_test)

y_pred_train = best_model.predict(X_train)
y_prob_train = best_model.predict_proba(X_train)

print(f'Akurasi Data Test: {tes_accuracy*100:.2f}%')

Hyperparameter Terbaik: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 1000, 'C': np.float64(0.008111308307896872)}
Akurasi Validasi Terbaik: 74.25%
Akurasi Data Test: 75.00%


In [270]:
def evaluate_model(pred,test,prob, evaluate, model_name='Logistic Regression'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test, pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:, 1])
    logloss = log_loss(test,prob)
    
    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }
    df_result = pd.DataFrame(data)
    return df_result

In [271]:
def check_model_fit(df_eval_train, df_eval_test):
    df_combined = pd.concat([df_eval_train, df_eval_test], ignore_index=True)
    acc_train = df_eval_train['Accuracy'].values[0]
    acc_test = df_eval_test['Accuracy'].values[0]
    gap = acc_train - acc_test

    if acc_train < 0.60 and acc_test < 0.60:
        status = 'Underfitting (akurasi rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f}%)'
    elif gap < -0.05:
        status = 'Overfitting / Data leakage (Test > Train)'
    else:
        status = 'Good fit'
    df_combined['Status'] = status
    return df_combined

In [272]:
df_eval_train = evaluate_model(y_pred_train, y_train, y_prob_train, evaluate='Training')
df_eval_test = evaluate_model(y_pred_test, y_test, y_prob_test, evaluate='Test')
df_eval = check_model_fit(df_eval_train, df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
                 Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Logistic Regression  Training  0.751667   0.747837  0.865522  0.802387  0.838934  0.519287  Good fit
1  Logistic Regression      Test  0.750000   0.765625  0.830508  0.796748  0.826512  0.531752  Good fit




## Melakukan Prediksi Dengan Data Random

In [273]:
from sklearn.preprocessing import StandardScaler
feauter_numerik = ['Usia','Tekanan_Darah_Rest','Kolesterol','Detak_Jantung_Max','Oldpeak_ST','BMI']
data_5_pasien = {
    'Usia': [63, 45, 56, 38, 50],
    'Jenis_Kelamin': [3, 1, 0, 1, 0],
    'Tipe_Nyeri_Dada': [-0.3, 1.20, 0.08, -1.05, -0.48],
    'Tekanan_Darah_Rest': [140, 130, 122, 110, 120],
    'Kolesterol': [69, 100, 122, 115, 190],
    'Gula_Darah_Puasa': [1, 0, 0, 0, 1],
    'EKG_Rest': [1, 2, 1, 1, 1],
    'Detak_Jantung_Max': [-0.98, 0.90, -1.05, 0.29, -1.58],
    'Angina_Olahraga': [44, 60, 56, 75, 14],
    'Oldpeak_ST': [19, 59, 73, 76, 18],
    'Kemiringan_ST': [1.28, 2.0, 1.5, 1.5, 2.0],
    'Jumlah_Pembuluh_Darah': [46, 60, 4, 0, 3],
    'Thalassemia': [81, 90, 1, 4, 1],
    'BMI': [10.77, 81.24, 24.5, 28.1, 29.3]
}
scaler = StandardScaler()
data_pasien= pd.DataFrame(data_5_pasien)
data_pasien[feauter_numerik] = scaler.fit_transform(data_pasien[feauter_numerik])
target_asli_pasien = [1, 0, 1, 0, 1] 

data_pasien_baru_x = data_pasien[X_train.columns]
data_pasien_baru_y = target_asli_pasien

In [274]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
# data_pasien_baru = df_x.sample(2, random_state=10)
data_pasien_baru = data_pasien
prediksi_pasien = best_model.predict(data_pasien_baru)
probabilitas_pasien = best_model.predict_proba(data_pasien_baru)

hasil_df = data_pasien_baru.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model.score(data_pasien_baru_x, data_pasien_baru_y)
prediksi_pasien = best_model.predict(data_pasien_baru_x)
probabilitas_pasien = best_model.predict_proba(data_pasien_baru_x)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))

hasil_df[['Peluang Aman(%)','Peluang Terkena (%)','Prediksi Serangan Jantung']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 60.00%

Tabel Skor Evaluasi Lengkap Data Baru:
              Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Logistic Regression Data_Baru       0.6        0.6     1.0      0.75      0.0  8.508057


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,4.658921e-07,100.000000,1
1,1.598239e-08,100.000000,1
2,1.373181e-05,99.999986,1
3,2.114357e-07,100.000000,1
4,8.784042e-01,99.121596,1
